In [1]:
import gurobipy as gp;
from gurobipy import GRB;
import numpy as np;

In [2]:
xi = [1,2,4];

numScens = len(xi);

# Define master problem
m = gp.Model("testLP");
m.modelSense = GRB.MINIMIZE; 

x = m.addVar(vtype=GRB.CONTINUOUS, lb = 0, ub = 10, obj=0);
theta = m.addVar(vtype=GRB.CONTINUOUS, lb = 0, ub = GRB.INFINITY, obj=1);

m.setParam("OutputFlag", 0); # disable output information unless you want to take a look at the solution process

Set parameter Username
Academic license - for non-commercial use only - expires 2025-07-29


In [3]:
# Just give an arbirtrary xval initially, will change iteratively anyway
xval = 0;

# Define subproblem
sub = gp.Model("subprob");
sub.modelSense = GRB.MINIMIZE; 

y1 = sub.addVar(vtype=GRB.CONTINUOUS, lb = 0, ub = GRB.INFINITY, obj=1);
y2 = sub.addVar(vtype=GRB.CONTINUOUS, lb = 0, ub = GRB.INFINITY, obj=1);
 
# Give a reference to each constraint since we will use the dual multiplier associated with each constraint
constr = sub.addConstr(y1 - y2 == 0);
# Just give an arbitrary rhs for now, will change iteratively anyway

sub.setParam("OutputFlag", 0); # disable output information unless you want to take a look at the solution process

In [4]:
LB = 0;
UB = 1e10;
iter = 0;

while (UB-LB)*1.0/(max(abs(UB),1e-8)) > 1e-5:
  iter = iter + 1;
  # First solve the master problem, and get x value
  m.optimize();
  xval = x.x;
  thetaval = theta.x;
  LB = m.objVal;

  duals = [0]*numScens;
  tempUB = 0;
  for k in range(numScens):  
    constr.setAttr(GRB.Attr.RHS, xi[k]-xval);
    # This is a simple case, since we have complete recourse: so there is no feasibility cuts
    sub.optimize();
    tempUB = tempUB + sub.objVal*1.0/numScens;
    duals[k] = constr.pi;

  if tempUB < UB:
    UB = tempUB;
  if (UB-LB)*1.0/(max(abs(UB),1e-8)) < 1e-5:
    break;
  else:
    # If not converge, MUST be able to generate a cut -- no need to check cut violation
    # Now construct cut:
    # Cut should look like: theta >= sum_k p_k(xi[k]-x)*dual[k]
    m.addConstr(theta + (sum(duals)*1.0/numScens)*x >= np.dot(duals,xi)*1.0/numScens)
    print('cut[%d]: cutcoef = %f, cutrhs = %f' % (iter, (sum(duals)*1.0/numScens), (np.dot(duals,xi)*1.0/numScens)));
    
print("LB = ", LB);
print("UB = ", UB);
print("xval = ", xval);

cut[1]: cutcoef = 1.000000, cutrhs = 2.333333
cut[2]: cutcoef = -0.333333, cutrhs = 0.333333
cut[3]: cutcoef = 0.333333, cutrhs = 1.666667
LB =  1.0
UB =  1.0
xval =  2.0000000000000004


# Object oriented

In [6]:
import gurobipy as gp;
from gurobipy import GRB;
import numpy as np;

In [7]:
# Class to handle scenario data
class ScenarioData:
    def __init__(self, scenarios):
        self.scenarios = scenarios
        self.num_scenarios = len(scenarios)

In [8]:
# Class for the master problem
class MasterProblem:
    def __init__(self):
        self.model = gp.Model("MasterProblem")
        self.model.setParam("OutputFlag", 0)
        self.x = self.model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=10, obj=0, name="x")
        self.theta = self.model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=GRB.INFINITY, obj=1, name="theta")
        self.model.modelSense = GRB.MINIMIZE
        self.model.update()

    def solve(self):
        self.model.optimize()
        return self.x.x, self.theta.x, self.model.objVal

    def add_cut(self, duals, scenarios, num_scenarios):
        """
        Add an optimality cut to the master problem.
        """
        cut_coef = sum(duals) / num_scenarios
        cut_rhs = np.dot(duals, scenarios) / num_scenarios
        self.model.addConstr(self.theta + cut_coef * self.x >= cut_rhs)
        self.model.update()

In [9]:
# Class for the subproblem
class SubProblem:
    def __init__(self):
        self.model = gp.Model("SubProblem")
        self.model.setParam("OutputFlag", 0)
        self.y1 = self.model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=GRB.INFINITY, obj=1, name="y1")
        self.y2 = self.model.addVar(vtype=GRB.CONTINUOUS, lb=0, ub=GRB.INFINITY, obj=1, name="y2")
        self.constraint = self.model.addConstr(self.y1 - self.y2 == 0, name="constr")
        self.model.modelSense = GRB.MINIMIZE
        self.model.update()

    def solve(self, rhs):
        self.constraint.setAttr(GRB.Attr.RHS, rhs)
        self.model.optimize()
        return self.model.objVal, self.constraint.pi

In [10]:
# Class for Benders decomposition
class BendersDecomposition:
    def __init__(self, scenario_data, master_problem, sub_problem, epsilon=1e-5):
        self.scenario_data = scenario_data
        self.master_problem = master_problem
        self.sub_problem = sub_problem
        self.LB = 0
        self.UB = 1e10
        self.iteration = 0
        self.epsilon = epsilon

    def run(self):
        while (self.UB - self.LB) / max(abs(self.UB), 1e-8) > self.epsilon:
            self.iteration += 1
    
            # Solve master problem
            xval, thetaval, obj_val = self.master_problem.solve()
            self.LB = obj_val
    
            # Print current iteration, LB, and UB
            print(f"Iteration {self.iteration}: LB = {self.LB:.6f}, UB = {self.UB:.6f}")
    
            # Solve subproblems
            duals = []
            tempUB = 0
            for idx in range(self.scenario_data.num_scenarios):  # Explicit index iteration
                scenario_value = self.scenario_data.scenarios[idx]  # Access by index
                rhs = scenario_value - xval
                sub_obj, dual = self.sub_problem.solve(rhs)
                tempUB += sub_obj / self.scenario_data.num_scenarios
                duals.append(dual)
    
            # Update UB
            if tempUB < self.UB:
                self.UB = tempUB
    
            # Check convergence
            if (self.UB - self.LB) / max(abs(self.UB), 1e-8) < self.epsilon:
                break
    
            # Add optimality cut
            self.master_problem.add_cut(duals, self.scenario_data.scenarios, self.scenario_data.num_scenarios)
            print(f"cut[{self.iteration}]: cutcoef = {sum(duals) / self.scenario_data.num_scenarios:.6f}, "
                  f"cutrhs = {np.dot(duals, self.scenario_data.scenarios) / self.scenario_data.num_scenarios:.6f}")
    
        print("#########################################################################################")
        print(f"Final LB = {self.LB:.6f}")
        print(f"Final UB = {self.UB:.6f}")
        print(f"xval = {xval}")

In [11]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from data import ScenarioData
from masterProblem import MasterProblem
from subProblem import SubProblem
from benders import BendersDecomposition

if __name__ == "__main__":
    scenarios = [1, 2, 4]
    scenario_data = ScenarioData(scenarios)
    master_problem = MasterProblem()
    sub_problem = SubProblem()
    benders = BendersDecomposition(scenario_data, master_problem, sub_problem)
    benders.run()

Iteration 1: LB = 0.000000, UB = 10000000000.000000
cut[1]: cutcoef = 1.000000, cutrhs = 2.333333
Iteration 2: LB = 0.000000, UB = 2.333333
cut[2]: cutcoef = -0.333333, cutrhs = 0.333333
Iteration 3: LB = 0.833333, UB = 1.111111
cut[3]: cutcoef = 0.333333, cutrhs = 1.666667
Iteration 4: LB = 1.000000, UB = 1.111111
#########################################################################################
Final LB = 1.000000
Final UB = 1.000000
xval = 2.0000000000000004
